## 1 · Install and load the package

In [26]:
# @title Install
!pip -q install google-genai pymupdf sentence-transformers langchain-text-splitters chromadb rank_bm25 pysqlite3-binary
print("Done.")

Done.


In [3]:
# @title Upload ragkit.zip and import
import os, sys, zipfile

if not os.path.exists("/content/ragkit/index.py"):
    from google.colab import files
    up = files.upload()                      # select ragkit.zip
    os.makedirs("/content/ragkit", exist_ok=True)
    with zipfile.ZipFile(list(up.keys())[0]) as z:
        z.extractall("/content/ragkit")

sys.path.insert(0, "/content/ragkit")

import sqlite3
if sqlite3.sqlite_version_info < (3, 35, 0):
    __import__("pysqlite3")
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

import extract, sections, index as idx, pipeline, evaluate as ev
print("ragkit loaded:", [m for m in ("extract","sections","index","pipeline","evaluate")])

Saving ragkit.zip to ragkit.zip
ragkit loaded: ['extract', 'sections', 'index', 'pipeline', 'evaluate']


## 2 · Gemini

Key goes in the Colab **Secrets** panel

In [5]:
# @title Gemini client + generate/ocr callables
import os
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass
assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY not set (Secrets panel)."

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.1-flash-lite"                # alias survives model retirements


def generate(prompt: str, temperature: float = 0.1) -> str:
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=temperature))
    return (r.text or "").strip()


def ocr(png_bytes: bytes) -> str:
    r = client.models.generate_content(
        model=MODEL,
        contents=[types.Part.from_bytes(data=png_bytes, mime_type="image/png"),
                  pipeline.OCR_PROMPT],
        config=types.GenerateContentConfig(temperature=0.0))
    return (r.text or "").strip()


print(generate("Reply with exactly: ready"))

ready


## 3 · Configure

`CFG` is passed to the index and read at call time, so changing a value and
re-running eval takes effect without rebuilding anything.

Bump `INDEX_VERSION` to start a fresh index. Never delete a Chroma directory
while a client holds it open — that is what produced `unable to open database
file` and `attempt to write a readonly database` previously.

In [6]:
# @title Config
INDEX_VERSION = 1
PDF_FOLDER   = "/content/pdfs"
DB_PATH      = f"/content/chroma_v{INDEX_VERSION}"
OCR_CACHE    = "/content/ocr_cache"          # survives restarts; never re-OCR
EMB_CACHE    = "/content/emb_cache"

CFG = {
    "embed_model": "BAAI/bge-m3",            # or intfloat/multilingual-e5-base
    "query_prefix": "", "doc_prefix": "",    # e5 needs "query: " / "passage: "
    "chunk_size": 700, "chunk_overlap": 120,
    "ngram": 4,                              # 0 = whole-word BM25
    "use_vector": True, "use_bm25": True, "use_rerank": True,
    "candidates": 60, "rrf_k": 60, "rerank_pool": 40,
    "top_k": 8, "dedupe": 0.85,
}

os.makedirs(PDF_FOLDER, exist_ok=True)
print(f"index v{INDEX_VERSION} -> {DB_PATH}")

index v1 -> /content/chroma_v1


## 4 · Add PDFs

In [7]:
# @title Upload PDFs (or copy them into PDF_FOLDER any other way)
from google.colab import files
import shutil

up = files.upload()
for name, data in up.items():
    open(os.path.join(PDF_FOLDER, name), "wb").write(data)

print("\nFolder contents:")
for f in sorted(os.listdir(PDF_FOLDER)):
    print(f"  {f}  ({os.path.getsize(os.path.join(PDF_FOLDER, f))//1024} KB)")

Saving lecture17.pdf to lecture17.pdf

Folder contents:
  lecture17.pdf  (77 KB)


In [8]:
# @title Inspect before ingesting: which pages will need OCR, and why
for f in sorted(os.listdir(PDF_FOLDER)):
    if not f.lower().endswith(".pdf"):
        continue
    q = extract.scan_pdf(os.path.join(PDF_FOLDER, f))
    need = [x for x in q if x.needs_ocr]
    print(f"\n{f}: {len(need)}/{len(q)} pages need OCR")
    for x in q[:3]:
        print(f"   p{x.page:>3} chars={x.n_chars:>5} malformed={x.malformed_ratio:.3f} "
              f"-> {x.reason}")


lecture17.pdf: 0/4 pages need OCR
   p  1 chars= 2618 malformed=0.000 -> text layer usable
   p  2 chars= 2160 malformed=0.000 -> text layer usable
   p  3 chars= 3017 malformed=0.000 -> text layer usable


## 5 · Build the index

OCR runs only on pages that need it, and is cached per page keyed by file
content hash. Embeddings are cached too. Re-running after a restart is cheap;
editing a PDF changes its hash and re-ingests only that file.

In [15]:
# @title Ingest folder -> index
index = idx.HybridIndex(db_path=DB_PATH, cfg=CFG, emb_cache=EMB_CACHE)
chunks = pipeline.build_corpus(PDF_FOLDER, index, ocr_fn=ocr, cache_dir=OCR_CACHE)
for c in index.corpus:
    c["section"] = "prose"

rag = pipeline.RAG(index, generate=generate)

print(f"\ncorpus: {len(index.corpus)} chunks across {len(index.docs)} documents")
print("sections:", sections.section_summary(index.corpus))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  lecture17.pdf: 4 pages, 0 need OCR 
    19 chunks {'question': 6, 'answer_key': 1, 'prose': 11, 'reference': 1}
  all 19 chunks already indexed

Indexed 19 chunks from 1 documents.
Documents: ['lecture17_a26d54f1']

corpus: 19 chunks across 1 documents
sections: {'prose': 19}


## 6 · Sanity checks

Run these before trusting any answer. If extraction is wrong, nothing
downstream can be right.

In [16]:
# @title Did the text come out clean?
import re, collections

# Malformed-mark ratio over the indexed corpus: should be ~0.
sample = "\n".join(c["text"] for c in index.corpus[:200])
print(f"malformed script ratio: {extract.malformed_ratio(sample):.4f}  (>0.05 = still broken)")

print("\nsection mix per document:")
per = collections.defaultdict(collections.Counter)
for c in index.corpus:
    per[c["doc"]][c["section"]] += 1
for doc, cnt in per.items():
    print(f"  {doc:<28} {dict(cnt)}")

print("\nsample chunk from each section:")
for sec in ("prose", "question", "answer_key", "reference"):
    hit = next((c for c in index.corpus if c["section"] == sec), None)
    if hit:
        print(f"\n[{sec}] {hit['doc']} p{hit['page']}")
        print("   ", hit["text"][:180].replace("\n", " "))

malformed script ratio: 0.0000  (>0.05 = still broken)

section mix per document:
  lecture17_a26d54f1           {'prose': 19}

sample chunk from each section:

[prose] lecture17_a26d54f1 p1
    Tangent, Cotangent, Secant, and Cosecant The Quotient Rule In our last lecture, among other things, we discussed the function 1 x, its domain and its derivative. We also showed how


## 7 · Gold set

Two sources, tried in order:

1. **The document's own answer keys.** Exam books, worksheets and FAQs print
   `উত্তর: খ` / `Answer: b` next to the options. Free and exactly on-distribution.
2. **LLM-generated** from prose chunks, for documents that print no key. The
   answer span is verified to appear verbatim in the source chunk, so a
   hallucinated pair is discarded rather than silently poisoning the metric.

Then filtering: an answer occurring 100+ times in the corpus gets "hit" by luck.
Those questions inflate every score without measuring retrieval.

In [17]:
# @title Build and filter the gold set
gold_keys = ev.gold_from_answer_keys(chunks)
print(f"{len(gold_keys)} pairs from printed answer keys")

gold_llm = []
if len(gold_keys) < 20:
    print("generating additional golds with the LLM ...")
    gold_llm = ev.gold_from_llm(chunks, generate, n_chunks=15, per_chunk=2)
    print(f"{len(gold_llm)} LLM-generated pairs (answer verified present in source)")

GOLD_ALL = gold_keys + gold_llm
GOLD = ev.filter_gold(GOLD_ALL, index.corpus, max_targets=3)

print()
for g in GOLD[:10]:
    print(f"  [{g['source']:<10}] {g['q'][:56]:<58} -> {g['answer'][:28]}")

0 pairs from printed answer keys
generating additional golds with the LLM ...
21 LLM-generated pairs (answer verified present in source)
gold filter: 4/21 kept (<= 3 targets; median targets was 5)

  [llm       ] What is the tangent function the quotient of?              -> sine over cosine
  [llm       ] What is cot x essentially?                                 -> 1 divided by the tangent
  [llm       ] What type of function is h(x)?                             -> rational polynomial function
  [llm       ] How many other trigonometric functions are discussed tod   -> four


In [18]:
# Data problem or ranking problem? (automatic)
# For each gold question, is its answer actually present in some chunk?
# If not, that item is a DATA problem — extraction/chunking lost it, and no
# ranking work will recover it. This needs no hand-picked pairs.
from evaluate import matches, target_counts

counts = target_counts(GOLD, index.corpus)
data_problems = [g for g, n in zip(GOLD, counts) if n == 0]

print(f"{len(GOLD)-len(data_problems)}/{len(GOLD)} gold answers present in the corpus")
if data_problems:
    print("\nDATA PROBLEMS (answer missing from every chunk):")
    for g in data_problems[:8]:
        print(f"  {g['q'][:60]}")
        print(f"     want: {g['answer']!r}  [{g.get('doc')} p{g.get('page')}]")
else:
    print("Every gold answer is retrievable. Any failure is a RANKING problem —")
    print("check MRR/hit@1 in the eval cell, not extraction.")

# For LLM golds, a stronger check: does the answer survive in ITS source chunk?
# If not, chunking split the fact from where the LLM found it.
llm_golds = [g for g in GOLD if g.get("gold_chunk")]
if llm_golds:
    by_id = {c["id"]: c for c in index.corpus}
    lost = [g for g in llm_golds
            if g["gold_chunk"] in by_id and not matches(g, by_id[g["gold_chunk"]]["text"])]
    print(f"\n{len(llm_golds)-len(lost)}/{len(llm_golds)} LLM golds still intact in their source chunk")

4/4 gold answers present in the corpus
Every gold answer is retrievable. Any failure is a RANKING problem —
check MRR/hit@1 in the eval cell, not extraction.

4/4 LLM golds still intact in their source chunk


## 8 · Measure

Two metrics kept separate:

- **retrieval** (MRR, hit@1/3/k) — did the right chunk reach the LLM, and how
  high did it rank? A MISS means the bug is upstream; prompt tuning cannot fix it.
- **accuracy** — did the LLM produce the right answer?

Always read them against the random baseline. "100% hit@8" on a 168-chunk
corpus where chance is 24% is a much weaker claim than it sounds.

In [19]:
# @title Baseline + end-to-end evaluation
chance = ev.random_baseline(index, GOLD)
print(f"random hit@{CFG['top_k']} baseline: {chance:.1%}\n")

metrics = ev.evaluate(index, GOLD, answer_fn=rag.ask, show=5)

random hit@8 baseline: 57.6%



Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

MRR 1.000 | hit@1 100% | hit@3 100% | hit@k 100% | acc 4/4 (100%)


In [20]:
# @title Ablation — what is each component actually worth?
# Retrieval-only: no API calls, so this is free to re-run.
rows = ev.ablate(index, GOLD)

variant                    MRR   hit@1   hit@3   hit@k
bm25 only                0.583     25%    100%    100%
vector only              0.875     75%    100%    100%
hybrid (RRF)             0.750     50%    100%    100%
hybrid, no ngram         0.708     50%    100%    100%
hybrid + rerank          1.000    100%    100%    100%


## 9 · Ask

`docs=[...]` restricts the search to particular files; omit it to search
everything. `return_sources=True` gives the chunks behind the answer, which is
what you want if you ever put a UI on this.

In [25]:
# @title Interactive
print("documents:", index.docs, "\n")
while True:
    q = input("\nQuestion (exit to quit): ").strip()
    if q.lower() in {"exit", "quit", ""}:
        break
    ans, srcs = rag.ask(q, debug=True, return_sources=True)
    print("\n" + ans)
    print("  sources:", [(c["doc"][:16], c["page"]) for c in srcs[:4]])

documents: ['lecture17_a26d54f1'] 


Question (exit to quit): কোশিয়েন্ট কি?

[retrieve] 'কোশিয়েন্ট কি?'
  lecture17_a26d54f1 p  2 [prose    ] rrf=0.0164 rr=+0.386 | by its cosine:
tan x = sin x
cos x.
The cotangent of x is deﬁned to be...
  lecture17_a26d54f1 p  3 [prose    ] rrf=0.0143 rr=+0.074 | If a = . . . , −3π
2 , π
2 , 5π
2 , . . . then the left vertical asymp...
  lecture17_a26d54f1 p  2 [prose    ] rrf=0.0149 rr=+0.042 | is
h′(x) = (x2 −4x + 5) · 1 −(x + 3) · (2x −4)
(x2 −4x + 5)2
= x2 −4x ...
  lecture17_a26d54f1 p  1 [prose    ] rrf=0.0161 rr=+0.023 | Tangent, Cotangent, Secant, and Cosecant
The Quotient Rule
In our last...
  lecture17_a26d54f1 p  2 [prose    ] rrf=0.0159 rr=+0.006 | asymptotes. You should verify that your sketches reﬂect these properti...
  lecture17_a26d54f1 p  3 [prose    ] rrf=0.0156 rr=+0.003 | if a is a value of x outside the domain of tan x, then
lim
x→a−tan x =...
  lecture17_a26d54f1 p  3 [prose    ] rrf=0.0154 rr=+0.003 | π instead of 2π. You can 

In [21]:
# @title Single question, scoped to one document
# ans = rag.ask("...", docs=[index.docs[0]])
ans, srcs = rag.ask("What is this document about?", return_sources=True)
print(ans)
print("\nsources:", [(c["doc"], c["page"]) for c in srcs])

This document is about the quotient rule for differentiation and the properties and derivatives of the trigonometric functions tangent, cotangent, secant, and cosecant.

sources: [('lecture17_a26d54f1', 2), ('lecture17_a26d54f1', 1), ('lecture17_a26d54f1', 2), ('lecture17_a26d54f1', 2), ('lecture17_a26d54f1', 1), ('lecture17_a26d54f1', 3), ('lecture17_a26d54f1', 3), ('lecture17_a26d54f1', 4)]
